In [1]:
import resource
import warnings
# import os
# os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
import torch

def enforce_hardware_limits(cpu_gb=25, gpu_gb=13):
    """
    Places strict hard limits on CPU and GPU memory allocation.
    Prevents silent kernel crashes by throwing catchable MemoryErrors instead.
    """
    # 1. Enforce CPU RAM Limit (Linux Resource Module)
    cpu_bytes = cpu_gb * 1024 * 1024 * 1024
    
    # RLIMIT_AS limits the maximum area (in bytes) of address space which may be taken by the process.
    # We set both the soft limit and hard limit to 25 GB.
    try:
        resource.setrlimit(resource.RLIMIT_AS, (cpu_bytes, cpu_bytes))
        print(f"✅ CPU RAM hard limit locked at {cpu_gb} GB.")
    except ValueError as e:
        print(f"⚠️ Could not set CPU limit: {e}")

    # 2. Enforce GPU VRAM Limit (PyTorch CUDA Memory Fraction)
    if torch.cuda.is_available():
        total_vram = torch.cuda.get_device_properties(0).total_memory
        requested_vram = gpu_gb * 1024 * 1024 * 1024
        
        # Calculate the fraction of total memory to allow
        fraction = requested_vram / total_vram
        
        if fraction < 1.0:
            torch.cuda.set_per_process_memory_fraction(fraction, 0)
            print(f"✅ GPU VRAM limit locked at {gpu_gb} GB ({fraction*100:.1f}% of hardware capacity).")
        else:
            actual_gb = total_vram / (1024**3)
            print(f"⚠️ Notice: Requested GPU limit ({gpu_gb} GB) exceeds actual hardware capability ({actual_gb:.1f} GB).")
            print(f"✅ GPU VRAM capped at max hardware limit (100%).")
            
        # Optional: Force PyTorch to release fragmented memory aggressively
        # import os
        # os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    else:
        print("⚠️ No CUDA GPU detected.")

# Run the enforcer!
enforce_hardware_limits(cpu_gb=29, gpu_gb=14)

✅ CPU RAM hard limit locked at 29 GB.
✅ GPU VRAM limit locked at 14 GB (96.1% of hardware capacity).


In [2]:
import pandas as pd

def build_overture_mappings(taxonomy_csv_path):
    """
    100% Dynamically builds competitor and synergy dictionaries.
    Zero hardcoded categories or regex matching.
    """
    print("Building Taxonomy Mappings from Official Overture Schema...")
    df = pd.read_csv(taxonomy_csv_path)
    
    # Clean strings safely to guarantee exact matching
    for col in ['New Primary Category', 'New Basic Level Category', 'Group (L0)']:
        df[col] = df[col].astype(str).str.lower().str.strip()
    
    valid_df = df[df['New Primary Category'] != 'nan']
    
    # -------------------------------------------------------------
    # A. COMPETITOR MAPPINGS (Automatically grouped by Basic Level Category)
    # -------------------------------------------------------------
    competitor_mappings = {}
    grouped_blc = valid_df.groupby('New Basic Level Category')['New Primary Category'].apply(lambda x: list(set(x))).to_dict()
    
    for blc, categories in grouped_blc.items():
        for cat in categories:
            competitor_mappings[cat] = categories
            
    # -------------------------------------------------------------
    # B. SYNERGY MAPPINGS (Automatically grouped by Group L0)
    # -------------------------------------------------------------
    synergy_mappings = {}
    
    # Dynamically extract every L0 Group (e.g., 'food_and_drink', 'health_care')
    l0_groups = [g for g in valid_df['Group (L0)'].unique() if g != 'nan']
    
    for group in l0_groups:
        # Grab every single primary category that falls under this L0 Group
        categories_in_group = valid_df[valid_df['Group (L0)'] == group]['New Primary Category'].tolist()
        synergy_mappings[group] = list(set(categories_in_group))
        
    print(f"-> Mapped {len(competitor_mappings)} competitor tags.")
    print(f"-> Generated {len(synergy_mappings)} automated synergy groups: {', '.join(l0_groups)}")
    
    return competitor_mappings, synergy_mappings

# Provide the path to the uploaded Overture CSV in Kaggle
TAXONOMY_CSV_PATH = '/kaggle/input/datasets/sciencekonstant/overture/categories.csv'

# Automatically generate both dictionaries
COMPETITOR_MAPPINGS, OVERTURE_TAXONOMY_MAPPINGS = build_overture_mappings(TAXONOMY_CSV_PATH)

Building Taxonomy Mappings from Official Overture Schema...
-> Mapped 2354 competitor tags.
-> Generated 13 automated synergy groups: arts_and_entertainment, services_and_business, community_and_government, cultural_and_historic, education, food_and_drink, geographic_entities, health_care, lifestyle_services, lodging, shopping, sports_and_recreation, travel_and_transportation


In [3]:
# !pip install duckdb pyarrow geopandas torch

In [4]:
import gc
# import torch
import numpy as np
import pandas as pd
import geopandas as gpd
import warnings
import re

warnings.filterwarnings('ignore')

class OvertureFeatureEngineGPU:
    def __init__(self, target_crs="EPSG:7755", batch_size=250_000_000):
        """
        EPSG:7755 (India Metric) ensures all distances are in exactly meters.
        batch_size prevents CUDA Out-Of-Memory errors during matrix multiplication.
        """
        self.crs = target_crs
        self.batch_size = batch_size
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"Engine Online. Core computation device: {self.device.type.upper()}")

    def fit(self, overture_dict):
        """
        Loads all Overture themes into the engine's memory.
        Expects a dict: {'places': gdf, 'roads': gdf, 'connectors': gdf, 'base': gdf}
        """
        print("Projecting Overture Themes to Metric System...")

        # Paranoia VRAM flush: guarantees previous block's tensors are destroyed
        torch.cuda.empty_cache()
        
        # Isolate projection memory: pop -> project -> delete -> collect
        tmp = overture_dict.pop('places', gpd.GeoDataFrame())
        self.places = tmp.to_crs(self.crs) if not tmp.empty else tmp
        del tmp; gc.collect()

        tmp = overture_dict.pop('roads', gpd.GeoDataFrame())
        self.roads = tmp.to_crs(self.crs) if not tmp.empty else tmp
        del tmp; gc.collect()

        tmp = overture_dict.pop('connectors', gpd.GeoDataFrame())
        self.connectors = tmp.to_crs(self.crs) if not tmp.empty else tmp
        del tmp; gc.collect()

        tmp = overture_dict.pop('base', gpd.GeoDataFrame())
        self.base = tmp.to_crs(self.crs) if not tmp.empty else tmp
        del tmp; gc.collect()

        # # 1. Pre-load GPU Tensors for blazing fast Point math
        print("Caching Point Clouds to VRAM...")
        # self.places_tensor = self._to_tensor(self.places) if not self.places.empty else torch.empty((0,2), device=self.device)
        # self.connectors_tensor = self._to_tensor(self.connectors) if not self.connectors.empty else torch.empty((0,2), device=self.device)
       # In fit(): Update empty tensor initializations and connector fallback
        self.places_tensor = self._to_tensor(self.places) if not self.places.empty else torch.empty((0,2), device=self.device, dtype=torch.float64)
        self.connectors_tensor = self._to_tensor(self.connectors) if not self.connectors.empty else torch.empty((0,2), device=self.device, dtype=torch.float64)

        # Segment Places by Taxonomy for O(1) filtering
        self.cat_tensors = {}
        # Dynamically find the category column based on Overture release version
        cat_col = next((c for c in ['primary_category', 'categories_primary', 'category', 'subtype'] 
                        if hasattr(self, 'places') and not self.places.empty and c in self.places.columns), None)
        
        if cat_col:
            # Normalize strings safely
            categories = self.places[cat_col].astype(str).str.lower().str.strip()
            for cat in categories.unique():
                mask = categories == cat
                self.cat_tensors[cat] = self._to_tensor(self.places[mask])
                

        # Bug
        # if self.connectors_tensor.shape[0] == 0 and hasattr(self, 'roads') and not self.roads.empty:
        #     print("Connectors missing. Extracting road vertices as proxy junctions...")
        #     def get_coords(geom):
        #         if geom is None or geom.is_empty: return []
        #         if geom.geom_type == 'LineString': return list(geom.coords)
        #         if geom.geom_type == 'MultiLineString': return [pt for line in geom.geoms for pt in line.coords]
        #         return []
            
        #     all_coords = [pt for geom in self.roads.geometry for pt in get_coords(geom)]
        # Improved
        if self.connectors_tensor.shape[0] == 0 and hasattr(self, 'roads') and not self.roads.empty:
            print("Connectors missing. Extracting road vertices as proxy junctions...")
            def get_coords(geom):
                if geom is None or geom.is_empty: return []
                # FIX: Force extraction of only X and Y (pt[:2]) to drop rogue Z-elevations
                if geom.geom_type == 'LineString': return [pt[:2] for pt in geom.coords]
                if geom.geom_type == 'MultiLineString': return [pt[:2] for line in geom.geoms for pt in line.coords]
                return []
            
            all_coords = [pt for geom in self.roads.geometry for pt in get_coords(geom)]
            if all_coords:
                # Use float64 to match query tensor dtype
                self.connectors_tensor = torch.tensor(all_coords, dtype=torch.float64, device=self.device)

        # # 2. Build CPU STRtrees for fast Polygon/Line intersections
        # print("Building CPU STRtrees for complex geometries...")
        # self.roads_sindex = self.roads.sindex if not self.roads.empty else None
        # self.base_sindex = self.base.sindex if not self.base.empty else None

        print("Engine Fit Complete. Ready for transformation.")
        # FREE HUGE CPU RAM: We don't need point geometries on the CPU anymore!
        if hasattr(self, 'places'): del self.places
        if hasattr(self, 'connectors'): del self.connectors
        gc.collect()

    # def _to_tensor(self, gdf):
    #     """Moves coordinates to GPU, using float64 to prevent catastrophic cancellation."""
    #     centroids = gdf.geometry.centroid
    #     coords = np.column_stack((centroids.x, centroids.y))
    #     coords = np.nan_to_num(coords, nan=np.inf)
        
    #     # CRITICAL FIX: Changed to float64 (Double Precision)
    #     # Prevents float32 cancellation when squaring EPSG:7755 coordinate millions.
    #     return torch.tensor(coords, dtype=torch.float64, device=self.device)

    # NEW CODE
    def _to_tensor(self, gdf):
        """Moves coordinates to GPU natively. Fast-paths Points to skip CPU GEOS overhead."""
        if gdf.empty:
            return torch.empty((0,2), device=self.device, dtype=torch.float64)
            
        # FAST PATH: If geometries are Points, bypass the heavy GEOS engine instantly
        if (gdf.geom_type == 'Point').all():
            coords = np.column_stack((gdf.geometry.x, gdf.geometry.y))
        else:
            # Fallback for Polygons/Lines
            centroids = gdf.geometry.centroid
            coords = np.column_stack((centroids.x, centroids.y))
            
        coords = np.nan_to_num(coords, nan=np.inf)
        return torch.tensor(coords, dtype=torch.float64, device=self.device)

    def _gpu_point_analytics(self, query_tensor, target_tensor, radii=[100, 300, 500, 1000]):
        """Computes multi-radius densities using float64 to ensure meter-level precision."""
        N_queries = query_tensor.shape[0]
        N_targets = target_tensor.shape[0]
        
        if N_targets == 0:
            return {r: np.zeros(N_queries, dtype=int) for r in radii}, \
                   np.full(N_queries, np.nan), np.zeros(N_queries, dtype=float)

        counts = {r: torch.zeros(N_queries, dtype=torch.int32, device=self.device) for r in radii}
        # Downcast outputs back to float32 to conserve memory
        nearest = torch.full((N_queries,), float('nan'), device=self.device, dtype=torch.float32)
        gravity = torch.zeros(N_queries, dtype=torch.float32, device=self.device)

        safe_batch = max(1, self.batch_size // max(1, N_targets))

        for i in range(0, N_queries, safe_batch):
            q_batch = query_tensor[i:i+safe_batch]
            
            # cdist automatically uses float64 here, restoring 1-meter precision
            dist = torch.cdist(q_batch, target_tensor)
            
            # Radii Counts: (dist > 1e-4) strictly ignores the query point matching itself
            for r in radii:
                counts[r][i:i+safe_batch] = ((dist > 1e-4) & (dist <= r)).sum(dim=1).to(torch.int32)
                
            if target_tensor.shape[0] > 1:
                top2_dist, _ = torch.topk(dist, k=2, dim=1, largest=False)
                is_self = top2_dist[:, 0] < 1e-4
                min_d = torch.where(is_self, top2_dist[:, 1], top2_dist[:, 0])
            elif target_tensor.shape[0] == 1:
                min_d = torch.where(dist[:, 0] < 1e-4, torch.tensor(float('nan'), device=self.device, dtype=torch.float64), dist[:, 0])
            else:
                min_d = torch.full((q_batch.shape[0],), float('nan'), device=self.device, dtype=torch.float64)

            min_d[min_d == float('inf')] = float('nan')
            nearest[i:i+safe_batch] = min_d.to(torch.float32)
            
            # Gravity Decay evaluates perfectly now with true meter distances
            decay = 1.0 / ((dist / 100.0)**2 + 1.0)
            decay[dist > 2000] = 0.0 
            gravity[i:i+safe_batch] = decay.sum(dim=1).to(torch.float32)
            
            # Instantly free the massive contiguous blocks back to the GPU pool
            del q_batch, dist, decay

        return {r: counts[r].cpu().numpy() for r in radii}, nearest.cpu().numpy(), gravity.cpu().numpy()

    def transform(self, query_batch, taxonomies, competitor_mappings=None):
        """Generates ML Features, mapping direct categories and synergies."""
        if query_batch.crs is None:
            raise ValueError("query_batch must have a defined CRS before processing.")
            
        targets = query_batch.to_crs(self.crs).copy()
        q_tensor = self._to_tensor(targets)
        
        # Normalize target categories to guarantee matching with dictionary keys
        if 'target_category' in targets.columns:
            targets['target_category'] = targets['target_category'].astype(str).str.lower().str.strip()
        else:
            targets['target_category'] = 'unknown'
        
        # 1. SCHEMA LOCK: Penalties strictly aligned to the 10000m max_distance threshold
        targets['comp_count_100m'] = 0
        targets['comp_count_300m'] = 0
        targets['comp_count_1000m'] = 0
        targets['comp_count_5000m'] = 0
        targets['nearest_comp_dist'] = 10000.0  
        targets['comp_gravity_score'] = 0.0
        
        for group in taxonomies.keys():
            targets[f'synergy_{group}_300m'] = 0
            targets[f'synergy_{group}_500m'] = 0
            targets[f'synergy_{group}_1000m'] = 0
            
        targets['junction_density_300m'] = 0
        targets['corner_lot_indicator'] = 0
        
        targets['nearest_road_class'] = 'unclassified'
        targets['nearest_road_surface'] = 'unknown'
        targets['dist_nearest_water'] = 10000.0 
        targets['dist_nearest_park'] = 10000.0

        # OLD CODE
        # # 2. GROUP 1: Direct Competition (Taxonomy-Aware with Robust Matching)
        # for cat in targets['target_category'].unique():
        #     mask = targets['target_category'] == cat
        #     cat_q_tensor = self._to_tensor(targets[mask])
            
        #     clean_cat = str(cat).lower().strip()
            
        #     if competitor_mappings and clean_cat in competitor_mappings:
        #         bucket = [str(c).lower().strip() for c in competitor_mappings[clean_cat]]
        #         # Context Restored: Wildcard match the bucket tags against VRAM tensor keys
        #         matched_keys = [k for k in self.cat_tensors.keys() if any(b in k for b in bucket)]
        #     else:
        #         # Context Restored: Fallback fuzzy match for unknown categories
        #         matched_keys = [k for k in self.cat_tensors.keys() if clean_cat in k or k in clean_cat]
                
        #     tensors = [self.cat_tensors[k] for k in matched_keys]
        #     t_tensor = torch.cat(tensors, dim=0) if tensors else torch.empty((0,2), device=self.device, dtype=torch.float64)
            
        #     counts, nearest, grav = self._gpu_point_analytics(cat_q_tensor, t_tensor, radii=[100, 300, 1000, 5000])
            
        #     targets.loc[mask, 'comp_count_100m'] = counts[100]
        #     targets.loc[mask, 'comp_count_300m'] = counts[300]
        #     targets.loc[mask, 'comp_count_1000m'] = counts[1000]
        #     targets.loc[mask, 'comp_count_5000m'] = counts[5000]
            
        #     targets.loc[mask, 'nearest_comp_dist'] = np.nan_to_num(nearest, nan=10000.0)
        #     targets.loc[mask, 'comp_gravity_score'] = grav

        # # 3. GROUP 2: Synergies (Robust Substring Matching Restored)
        # for group_name, cat_list in taxonomies.items():
        #     clean_cat_list = [str(c).lower().strip() for c in cat_list]
            
        #     # Context Restored: Substring match to survive Overture tag drift
        #     matched_keys = [
        #         k for k in self.cat_tensors.keys() 
        #         if any(keyword in k for keyword in clean_cat_list)
        #     ]
            
        #     tensors = [self.cat_tensors[k] for k in matched_keys]
            
        #     if tensors:
        #         combined_target = torch.cat(tensors, dim=0)
        #         counts, _, _ = self._gpu_point_analytics(q_tensor, combined_target, radii=[300, 500, 1000])
        #         targets[f'synergy_{group_name}_300m'] = counts[300]
        #         targets[f'synergy_{group_name}_500m'] = counts[500]
        #         targets[f'synergy_{group_name}_1000m'] = counts[1000]

        # NEW CODE
        # 2. GROUP 1: Direct Competition (Exact Set Intersection)
        for cat in targets['target_category'].unique():
            mask = targets['target_category'] == cat
            cat_q_tensor = self._to_tensor(targets[mask])
            clean_cat = str(cat).lower().strip()
            
            # 1. Get the bucket of true competitors
            bucket_list = competitor_mappings.get(clean_cat, [clean_cat]) if competitor_mappings else [clean_cat]
            bucket_set = set([str(c).lower().strip() for c in bucket_list])
            
            # 2. EXACT INTERSECTION: Prevents "bar" from matching "barber_shop" 
            # while safely handling stringified arrays like "['cafe', 'bakery']"
            matched_keys = [
                k for k in self.cat_tensors.keys() 
                if set(re.findall(r'[a-z_]+', k)).intersection(bucket_set)
            ]
            
            tensors = [self.cat_tensors[k] for k in matched_keys]
            t_tensor = torch.cat(tensors, dim=0) if tensors else torch.empty((0,2), device=self.device, dtype=torch.float64)
            
            counts, nearest, grav = self._gpu_point_analytics(cat_q_tensor, t_tensor, radii=[100, 300, 1000, 5000])
            
            targets.loc[mask, 'comp_count_100m'] = counts[100]
            targets.loc[mask, 'comp_count_300m'] = counts[300]
            targets.loc[mask, 'comp_count_1000m'] = counts[1000]
            targets.loc[mask, 'comp_count_5000m'] = counts[5000]
            
            targets.loc[mask, 'nearest_comp_dist'] = np.nan_to_num(nearest, nan=10000.0)
            targets.loc[mask, 'comp_gravity_score'] = grav

        # 3. GROUP 2: Synergies (Exact Set Intersection)
        for group_name, cat_list in taxonomies.items():
            bucket_set = set([str(c).lower().strip() for c in cat_list])
            
            matched_keys = [
                k for k in self.cat_tensors.keys() 
                if set(re.findall(r'[a-z_]+', k)).intersection(bucket_set)
            ]
            
            tensors = [self.cat_tensors[k] for k in matched_keys]
            
            if tensors:
                combined_target = torch.cat(tensors, dim=0)
                counts, _, _ = self._gpu_point_analytics(q_tensor, combined_target, radii=[300, 500, 1000])
                targets[f'synergy_{group_name}_300m'] = counts[300]
                targets[f'synergy_{group_name}_500m'] = counts[500]
                targets[f'synergy_{group_name}_1000m'] = counts[1000]


       # 4. GROUP 3: Traffic & Morphology
        if self.connectors_tensor.shape[0] > 0:
            conn_counts, _, _ = self._gpu_point_analytics(q_tensor, self.connectors_tensor, radii=[25, 300])
            targets['junction_density_300m'] = conn_counts[300]
            targets['corner_lot_indicator'] = (conn_counts[25] > 0).astype(int) 

        
        # OLD CODE
       #  # 5. GROUP 4 & 5: Environment & Roads
       #  if hasattr(self, 'roads') and not self.roads.empty:
       #      r_class_col = next((c for c in ['class', 'highway', 'type', 'subtype'] if c in self.roads.columns), None)
       #      r_surf_col = next((c for c in ['surface', 'road_surface'] if c in self.roads.columns), None)
            
       #      keep_cols = ['geometry']
       #      if r_class_col: keep_cols.append(r_class_col)
       #      if r_surf_col: keep_cols.append(r_surf_col)
                
       #      r_near = gpd.sjoin_nearest(targets[['geometry']], self.roads[keep_cols], how='left', max_distance=10000)
       #      r_near = r_near[~r_near.index.duplicated(keep='first')]
            
       #      if r_class_col and r_class_col in r_near.columns:
       #          targets['nearest_road_class'] = r_near[r_class_col].fillna('unclassified').astype(str)
       #      if r_surf_col and r_surf_col in r_near.columns:
       #          targets['nearest_road_surface'] = r_near[r_surf_col].fillna('unknown').astype(str

        # if hasattr(self, 'base') and not self.base.empty:
        #     # CRITICAL FIX: include=['category'] so RAM-compressed text columns are visible
        #     text_cols = self.base.select_dtypes(include=['object', 'string', 'category']).columns
            
        #     if len(text_cols) > 0:
        #         base_text = self.base[text_cols].fillna('').apply(lambda x: ' '.join(x.astype(str).str.lower()), axis=1)
                
        #         water = self.base[base_text.str.contains('water|river|lake|pond|ocean|stream')]
        #         parks = self.base[base_text.str.contains('park|forest|nature|grass|wood|meadow')]
                
        #         if not water.empty:
        #             w_near = gpd.sjoin_nearest(targets[['geometry']], water, how='left', distance_col='d_water', max_distance=10000)
        #             w_near = w_near[~w_near.index.duplicated(keep='first')]
        #             targets['dist_nearest_water'] = w_near['d_water'].fillna(10000.0)
                
        #         if not parks.empty:
        #             p_near = gpd.sjoin_nearest(targets[['geometry']], parks, how='left', distance_col='d_park', max_distance=10000)
        #             p_near = p_near[~p_near.index.duplicated(keep='first')]
        #             targets['dist_nearest_park'] = p_near['d_park'].fillna(10000.0)

        # 5. GROUP 4 & 5: Environment & Roads
        if hasattr(self, 'roads') and not self.roads.empty:
            r_class_col = next((c for c in ['class', 'highway', 'type', 'subtype'] if c in self.roads.columns), None)
            r_surf_col = next((c for c in ['surface', 'road_surface'] if c in self.roads.columns), None)
            
            keep_cols = ['geometry']
            if r_class_col: keep_cols.append(r_class_col)
            if r_surf_col: keep_cols.append(r_surf_col)
                
            r_near = gpd.sjoin_nearest(targets[['geometry']], self.roads[keep_cols], how='left', max_distance=10000)
            r_near = r_near[~r_near.index.duplicated(keep='first')]

            # OLD CODE
            # if r_class_col and r_class_col in r_near.columns:
            #     targets['nearest_road_class'] = r_near[r_class_col].fillna('unclassified').astype(str)
            # if r_surf_col and r_surf_col in r_near.columns:
            #     targets['nearest_road_surface'] = r_near[r_surf_col].fillna('unknown').astype(str)
        
            # NEW CODE
            if r_class_col and r_class_col in r_near.columns:
                targets['nearest_road_class'] = r_near[r_class_col].astype(object).fillna('unclassified').astype(str)
            if r_surf_col and r_surf_col in r_near.columns:
                targets['nearest_road_surface'] = r_near[r_surf_col].astype(object).fillna('unknown').astype(str)

        if hasattr(self, 'base') and not self.base.empty and 'subtype' in self.base.columns:
            # CPU SAVER: Vectorized exact column search instead of slow row-by-row string concatenation
            subtypes = self.base['subtype'].astype(str).str.lower()
            
            water = self.base[subtypes.str.contains('water|river|lake|pond|ocean|stream')]
            parks = self.base[subtypes.str.contains('park|forest|nature|grass|wood|meadow')]
            
            if not water.empty:
                w_near = gpd.sjoin_nearest(targets[['geometry']], water, how='left', distance_col='d_water', max_distance=10000)
                w_near = w_near[~w_near.index.duplicated(keep='first')]
                targets['dist_nearest_water'] = w_near['d_water'].fillna(10000.0)
            
            if not parks.empty:
                p_near = gpd.sjoin_nearest(targets[['geometry']], parks, how='left', distance_col='d_park', max_distance=10000)
                p_near = p_near[~p_near.index.duplicated(keep='first')]
                targets['dist_nearest_park'] = p_near['d_park'].fillna(10000.0)
        
        return targets.to_crs("EPSG:4326")

In [ ]:
## import geopandas as gpd
import pandas as pd
import numpy as np
import gc
from shapely.geometry import LineString
import warnings
import pyarrow.parquet as pq
import pyarrow.dataset as ds
import re
import concurrent.futures
import shapely
import pyarrow as pa
import pyarrow.compute as pc

warnings.filterwarnings('ignore')

# --- 1. SET YOUR PATHS ---
PLACES_PATH = '/kaggle/input/datasets/sciencekonstant/overdata1/india_places.parquet'
ROADS_PATH = '/kaggle/input/datasets/sciencekonstant/overdata2/india_roads.parquet'
CONN_PATH = '/kaggle/input/datasets/sciencekonstant/overdata2/india_connectors.parquet'
BASE_PATH = '/kaggle/input/datasets/sciencekonstant/overdata4/india_base.parquet'

# --- 2. LOAD TARGETS (Fits in RAM) ---
print("Loading Target Locations...")
places = gpd.read_parquet(PLACES_PATH, columns=['primary_category', 'geometry'])
places.set_crs("EPSG:4326", allow_override=True, inplace=True)

# NEW: Read Parquet Metadata exactly ONCE globally to prevent I/O spam
print("Reading Parquet Metadata...")
ROADS_DS = ds.dataset(ROADS_PATH, format="parquet")
CONN_DS = ds.dataset(CONN_PATH, format="parquet")
BASE_DS = ds.dataset(BASE_PATH, format="parquet")

# --- 3. CREATE SPATIAL GRID ---
grid_size=1.5
print("Creating Geographic Grid...")
places['grid_x'] = np.floor(places.geometry.x / grid_size) * grid_size
places['grid_y'] = np.floor(places.geometry.y / grid_size) * grid_size

grouped = places.groupby(['grid_x', 'grid_y'])
print(f"Divided India into {len(grouped)} processing blocks.")
# --- 3. CREATE SPATIAL GRID WITH DYNAMIC CAPACITY LIMITS ---
# INITIAL_GRID_SIZE = 1.5
# MAX_TARGETS_PER_BLOCK = 50_000  # <-- TUNE THIS: Lower this if Kaggle still hits OOM

# print("Creating Geographic Grid...")

# def partition_spatially(gdf, current_grid_size, max_targets):
#     # Base case: chunk is small enough OR grid is getting too tiny (< ~500 meters)
#     # The minimum grid size prevents infinite recursion if 30k businesses have the exact same GPS coordinate.
#     if len(gdf) <= max_targets or current_grid_size < 0.005:
#         return [gdf]
    
#     # Split into 4 smaller quadrants by halving the grid size
#     new_grid_size = current_grid_size / 2.0
#     gdf = gdf.copy()
#     gdf['grid_x'] = np.floor(gdf.geometry.x / new_grid_size) * new_grid_size
#     gdf['grid_y'] = np.floor(gdf.geometry.y / new_grid_size) * new_grid_size
    
#     chunks = []
#     for _, sub_gdf in gdf.groupby(['grid_x', 'grid_y']):
#         chunks.extend(partition_spatially(sub_gdf, new_grid_size, max_targets))
#     return chunks

# # 1. Initial coarse grouping (Fast)
# places['grid_x'] = np.floor(places.geometry.x / INITIAL_GRID_SIZE) * INITIAL_GRID_SIZE
# places['grid_y'] = np.floor(places.geometry.y / INITIAL_GRID_SIZE) * INITIAL_GRID_SIZE

# # 2. Recursively split over-dense chunks
# final_blocks = []
# for _, target_chunk in places.groupby(['grid_x', 'grid_y']):
#     final_blocks.extend(partition_spatially(target_chunk, INITIAL_GRID_SIZE, MAX_TARGETS_PER_BLOCK))

# # 3. Format as a list of tuples to perfectly match your existing loop's destructuring:
# #    for i, ((gx, gy), target_chunk) in enumerate(grouped):
# grouped = [
#     ((chunk['grid_x'].iloc[0], chunk['grid_y'].iloc[0]), chunk) 
#     for chunk in final_blocks if not chunk.empty
# ]

# print(f"Divided India into {len(grouped)} dynamic processing blocks (Max limit: {MAX_TARGETS_PER_BLOCK} targets/block).")

# Initialize the Engine
# engine = OvertureFeatureEngineGPU(target_crs="EPSG:7755", batch_size=100_000)

# Change from 100_000 to 500_000_000
engine = OvertureFeatureEngineGPU(target_crs="EPSG:7755", batch_size=5_000_000)

BASE_PATTERN = re.compile(r'water|river|lake|pond|ocean|stream|park|forest|nature|grass|wood|meadow', re.IGNORECASE)

# # --- 4. THE MEMORY-SAFE LOOP ---
# for i, ((gx, gy), target_chunk) in enumerate(grouped):
#     print(f"\n--- Processing Block {i+1}/{len(grouped)} (Grid: {gx}, {gy}) ---")

from itertools import islice

# The index you want to start/resume from
start_i = 212  

# --- 4. THE MEMORY-SAFE LOOP ---
# islice(grouped, start_i, None) skips the first 'start_i' groups.
# start=start_i in enumerate ensures 'i' matches your true position.
for i, ((gx, gy), target_chunk) in enumerate(islice(grouped, start_i, None), start=start_i):
    # Your processing code here...
    print(f"\n--- Processing Block {i+1}/{len(grouped)} (Grid: {gx}, {gy}) ---")
    print(f"Target businesses in this block: {len(target_chunk)}")
    
    # A. Buffer expanded to 0.05 degrees (~5.5 km) to support 5000m radii
    minx, miny, maxx, maxy = target_chunk.total_bounds
    bbox = (minx - 0.05, miny - 0.05, maxx + 0.05, maxy + 0.05)
    
    # B. Slice local places from the RAM dataframe
    local_places = places.cx[bbox[0]:bbox[2], bbox[1]:bbox[3]].copy()

    # NEW C SECTION
    # C. LOAD LOCAL CONTEXT SAFELY & FAST (Vectorized Streaming)

    # NEW C SECTION
    # C. LOAD LOCAL CONTEXT SAFELY & FAST (Vectorized Streaming)

    # def stream_layer(path, bounds, is_base=False):
    # def stream_layer(path, bounds, is_base=False):
    def stream_layer(dataset, bounds, is_base=False, is_roads=False):
        minx, miny, maxx, maxy = bounds
        try:
            # dataset = ds.dataset(path, format="parquet")
            
            # # Fetch only the columns we actually need to save RAM
            # valid_cols = dataset.schema.names
            # if not is_base:
            #     keep = ['class', 'surface', 'geometry'] if 'roads' in path else ['geometry']
            #     valid_cols = [c for c in valid_cols if c in keep]

            # OLD CODE
            # Fetch only the columns we actually need to save RAM
            # schema_cols = dataset.schema.names
            # if is_base:
            #     valid_cols = [c for c in schema_cols if c in ['subtype', 'geometry']]
            # elif 'roads' in path:
            #     valid_cols = [c for c in schema_cols if c in ['class', 'surface', 'geometry']]
            # else:
            #     valid_cols = [c for c in schema_cols if c in ['geometry']]
            # NEW CODE 
            # Fetch only the columns we actually need to save RAM
            schema_cols = dataset.schema.names
            if is_base:
                valid_cols = [c for c in schema_cols if c in ['subtype', 'geometry']]
            elif is_roads:
                valid_cols = [c for c in schema_cols if c in ['class', 'surface', 'geometry']]
            else:
                valid_cols = [c for c in schema_cols if c in ['geometry']]
                
            valid_chunks = []

            # OLD CODE 1
            # # batch_size=50_000 drastically reduces Pandas concatenation overhead
            # for batch in dataset.to_batches(columns=valid_cols, batch_size=250_000):
            #     df_batch = batch.to_pandas()
                
            #     # CPU SAVER: Drop 95% of Base geometries BEFORE the heavy WKB parser
            #     # if is_base and 'subtype' in df_batch.columns:
            #     #     mask = df_batch['subtype'].astype(str).str.lower().str.contains('water|river|lake|pond|ocean|stream|park|forest|nature|grass|wood|meadow')
            #     if is_base and 'subtype' in df_batch.columns:
            #         mask = df_batch['subtype'].str.contains(BASE_PATTERN, na=False)
            #         df_batch = df_batch[mask]
            #         if df_batch.empty: continue

            for batch in dataset.to_batches(columns=valid_cols, batch_size=100_000):
                # 1. C++ text filter: Drop non-water/park rows directly in Arrow BEFORE converting to Pandas
                if is_base and 'subtype' in schema_cols:
                    mask = pc.match_substring_regex(batch.column('subtype'), "(?i)water|river|lake|pond|ocean|stream|park|forest|nature|grass|wood|meadow")
                    batch = batch.filter(pc.fill_null(mask, False))
                    if batch.num_rows == 0: continue

                # OLDER PYTHON CODE
                    
                # # Parse WKB safely
                # geoms = gpd.GeoSeries.from_wkb(df_batch['geometry'], on_invalid='ignore')
                # valid_mask = geoms.notna() & ~geoms.is_empty
                # if not valid_mask.any(): continue
                    
                # # # Bounding Box Filter
                # # centroids = geoms[valid_mask].centroid
                # # x, y = centroids.x, centroids.y
                # # spatial_mask = (x >= minx) & (x <= maxx) & (y >= miny) & (y <= maxy)
                # # Bounding Box Filter
                
                # # OLD CODE
                # # centroids = geoms[valid_mask].centroid
                # # cx, cy = centroids.x.values, centroids.y.values
                # # spatial_mask = (cx >= minx) & (cx <= maxx) & (cy >= miny) & (cy <= maxy)
                # # 4. CPU & QUALITY FIX: Use .bounds instead of .centroid. 
                # # Avoids calculus overhead and correctly keeps long roads/parks that clip the edges of the box.
                # bounds = geoms[valid_mask].bounds
                # spatial_mask = (
                #     (bounds['maxx'].values >= minx) & 
                #     (bounds['minx'].values <= maxx) & 
                #     (bounds['maxy'].values >= miny) & 
                #     (bounds['miny'].values <= maxy)
                # )

                # # OLD
                # # final_mask = valid_mask.copy()
                # # final_mask.loc[valid_mask] = spatial_mask
                
                # # if final_mask.any():

                # # NEW: Clean boolean intersection
                # final_mask = valid_mask.copy()
                # final_mask[valid_mask] = spatial_mask
                
                # if final_mask.any():
                #     chunk = df_batch.loc[final_mask].copy()
                #     chunk['geometry'] = geoms[final_mask]

                #     # OLD
                #     # # RAM SAVER: Compress text to categories instantly
                #     # for col in chunk.columns:
                #     #     if col != 'geometry' and chunk[col].dtype == 'object':
                #     #         chunk[col] = chunk[col].astype(str).astype('category')

                # ---------------------------------------------------------
                # DRASTIC CPU OPTIMIZATION: Pure Shapely C-API & NumPy Math
                # ---------------------------------------------------------

                # OLD CODE 2
                # # Parse WKB directly to C-pointers (bypasses Pandas allocation overhead)
                # raw_geoms = shapely.from_wkb(df_batch['geometry'].values)
                
                # 2. Extract ONLY the geometry array directly from Arrow (no full dataframe created yet)
                raw_geoms = shapely.from_wkb(batch.column('geometry').to_pandas().values)
                
                # Fast NumPy boolean masking
                valid_mask = ~(shapely.is_missing(raw_geoms) | shapely.is_empty(raw_geoms))
                if not valid_mask.any(): continue
                    
                valid_geoms = raw_geoms[valid_mask]
                
                # Bounds returns a fast C-level NumPy array of shape (N, 4): [minx, miny, maxx, maxy]
                b = shapely.bounds(valid_geoms)
                
                # Vectorized bitwise spatial math via NumPy columns
                spatial_mask = (b[:, 2] >= minx) & (b[:, 0] <= maxx) & (b[:, 3] >= miny) & (b[:, 1] <= maxy)

                # Clean boolean intersection
                final_mask = valid_mask.copy()
                final_mask[valid_mask] = spatial_mask
                
                if final_mask.any():
                    # OLD CODE 3
                    # chunk = df_batch.loc[final_mask].copy()
                    # 3. Convert ONLY the surviving 1% of rows into Pandas
                    chunk = batch.filter(pa.array(final_mask)).to_pandas()
                    # Re-attach as GeoPandas object ONLY for the surviving geometries
                    chunk['geometry'] = gpd.GeoSeries(raw_geoms[final_mask], index=chunk.index)
                            
                    valid_chunks.append(chunk)
                    
            if not valid_chunks:
                return gpd.GeoDataFrame(columns=valid_cols, geometry='geometry', crs="EPSG:4326")
                
            # return gpd.GeoDataFrame(pd.concat(valid_chunks, ignore_index=True), geometry='geometry', crs="EPSG:4326")
            # Concat first, then compress. O(1) dictionary creation instead of O(N) unions.
            final_df = pd.concat(valid_chunks, ignore_index=True)
            for col in final_df.columns:
                if col != 'geometry' and final_df[col].dtype == 'object':
                    final_df[col] = final_df[col].astype(str).astype('category')
            
            return gpd.GeoDataFrame(final_df, geometry='geometry', crs="EPSG:4326")
            
        except Exception as e:
            print(f"Skipping layer: {e}")
            return gpd.GeoDataFrame(columns=['geometry'], geometry='geometry', crs="EPSG:4326")

    # # Stream the layers dynamically (Nothing is globally cached)
    # local_roads = stream_layer(ROADS_PATH, bbox, is_base=False)
    # local_connectors = stream_layer(CONN_PATH, bbox, is_base=False)
    # local_base = stream_layer(BASE_PATH, bbox, is_base=True)
    # Stream the layers dynamically using 3 CPU cores in parallel
    with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
        # OLD
        # future_roads = executor.submit(stream_layer, ROADS_PATH, bbox, False)
        # future_conn = executor.submit(stream_layer, CONN_PATH, bbox, False)
        # future_base = executor.submit(stream_layer, BASE_PATH, bbox, True)
        # NEW
        future_roads = executor.submit(stream_layer, ROADS_DS, bbox, is_base=False, is_roads=True)
        future_conn = executor.submit(stream_layer, CONN_DS, bbox, is_base=False, is_roads=False)
        future_base = executor.submit(stream_layer, BASE_DS, bbox, is_base=True, is_roads=False)

        # Wait for all 3 threads to finish and grab the results
        local_roads = future_roads.result()
        local_connectors = future_conn.result()
        local_base = future_base.result()

    # E. Set CRSs safely
    # for layer in [local_places, local_roads, local_connectors, local_base]:
    #     layer.set_crs("EPSG:4326", allow_override=True, inplace=True)
        
    # F. Feed to Engine
    overture_dict = {
        'places': local_places,
        'roads': local_roads,
        'connectors': local_connectors,
        'base': local_base
    }
    
    engine.fit(overture_dict)
    
    # G. Generate Features
    query_gdf = target_chunk.copy()
    query_gdf['target_category'] = query_gdf['primary_category']
    
    # # BUSINESS_SYNERGY_RULES is loaded from your Cell 1 taxonomy
    features = engine.transform(query_gdf, OVERTURE_TAXONOMY_MAPPINGS, COMPETITOR_MAPPINGS)
    
    # H. Save chunk to disk incrementally to prevent Kaggle disk/metadata corruption
    out_name = f'features_chunk_{i}.parquet'
    features.drop(columns=['grid_x', 'grid_y']).to_parquet(out_name, compression='zstd')
    print(f"Saved: {out_name}")
    
   # I. Obliterate RAM completely before the next loop
    del local_places, local_roads, local_connectors, local_base, overture_dict, query_gdf, features
    
    # 1. Force Python to wipe CPU RAM
    gc.collect() 
    
    # 2. Force PyTorch to wipe GPU VRAM (PREVENTS CUDA OOM!)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n🚀 Success! ML Feature Matrix completed without breaching RAM limits.")

Loading Target Locations...
Reading Parquet Metadata...
Creating Geographic Grid...
Divided India into 308 processing blocks.
Engine Online. Core computation device: CUDA

--- Processing Block 213/308 (Grid: 87.0, 24.0) ---
Target businesses in this block: 10650
Projecting Overture Themes to Metric System...
Caching Point Clouds to VRAM...
Engine Fit Complete. Ready for transformation.
Saved: features_chunk_212.parquet

--- Processing Block 214/308 (Grid: 87.0, 25.5) ---
Target businesses in this block: 18681
Projecting Overture Themes to Metric System...
Caching Point Clouds to VRAM...
Engine Fit Complete. Ready for transformation.
Saved: features_chunk_213.parquet

--- Processing Block 215/308 (Grid: 87.0, 27.0) ---
Target businesses in this block: 3849
Projecting Overture Themes to Metric System...
Caching Point Clouds to VRAM...
Engine Fit Complete. Ready for transformation.
Saved: features_chunk_214.parquet

--- Processing Block 216/308 (Grid: 87.0, 28.5) ---
Target businesses in 